# Amharic News Topic Classifier

**Goal:** Fine-tune a transformer to classify Amharic news into 6 topics.

**Dataset:** [israel/Amharic-News-Text-classification-Dataset](https://huggingface.co/datasets/israel/Amharic-News-Text-classification-Dataset) (~51k real articles from Ethiopian news sites).

**Colab tip:** Runtime → Change runtime type → **T4 GPU**.

## Phase 1 — Setup

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — enable GPU in Colab!")

## Phase 2 — Load data

The dataset already has **train** and **test** splits. Labels come from news-site categories (objective, not AI-generated).

In [ ]:
DATASET_NAME = "israel/Amharic-News-Text-classification-Dataset"
MODEL_NAME = "Davlan/afro-xlmr-base"  # Africa-focused XLM-R; good for Amharic

raw = load_dataset(DATASET_NAME)
raw

## Phase 3 — Quick look

Six classes: politics, sport, business, local news, international news, entertainment.

In [ ]:
from collections import Counter

counts = Counter(raw["train"]["category"])
for label, n in counts.most_common():
    print(f"{label:20} {n:,}")

print("\nExample:")
row = raw["train"][0]
print("Category:", row["category"])
print("Headline:", row["headline"])
print("Article (first 200 chars):", row["article"][:200])

## Phase 4 — Prepare text & labels

We combine **headline + article** so the model sees full context. Long texts are truncated later during tokenization.

In [ ]:
MAX_LENGTH = 256  # increase to 384/512 if you have more GPU time

# Map Amharic labels → readable English names (for reports & the web app later)
LABEL2ID = {
    "ሀገር አቀፍ ዜና": 0,
    "ስፖርት": 1,
    "ፖለቲካ": 2,
    "ዓለም አቀፍ ዜና": 3,
    "ቢዝነስ": 4,
    "መዝናኛ": 5,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
EN_LABELS = {
    0: "local_news",
    1: "sport",
    2: "politics",
    3: "international_news",
    4: "business",
    5: "entertainment",
}


def make_text(example):
    headline = (example["headline"] or "").strip()
    article = (example["article"] or "").strip()
    example["text"] = f"{headline}. {article}" if headline else article
    example["label"] = LABEL2ID[example["category"]]
    return example


dataset = raw.map(make_text, remove_columns=raw["train"].column_names)
dataset

## Phase 5 — Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized = dataset.map(tokenize, batched=True)
tokenized

## Phase 6 — Train

Expect ~1–2 hours on Colab free T4. Published benchmarks on this data reach **~88–90% F1** with similar models.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": (preds == labels).mean(),
        "f1_weighted": metric.compute(
            predictions=preds, references=labels, average="weighted"
        )["f1"],
        "f1_macro": metric.compute(
            predictions=preds, references=labels, average="macro"
        )["f1"],
    }


args = TrainingArguments(
    output_dir="./checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## Phase 7 — Evaluate on test set

In [ ]:
results = trainer.evaluate(tokenized["test"])
print(f"Accuracy:    {results['eval_accuracy']:.4f}")
print(f"F1 weighted: {results['eval_f1_weighted']:.4f}")
print(f"F1 macro:    {results['eval_f1_macro']:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

preds = trainer.predict(tokenized["test"]).predictions.argmax(axis=-1)
labels = tokenized["test"]["label"]

target_names = [EN_LABELS[i] for i in range(len(EN_LABELS))]
print(classification_report(labels, preds, target_names=target_names))

cm = confusion_matrix(labels, preds)
pd.DataFrame(cm, index=target_names, columns=target_names)

## Phase 8 — Save model (for Streamlit app later)

In [ ]:
SAVE_DIR = "amharic-news-classifier-model"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save label maps so the web app can read them
import json

with open(f"{SAVE_DIR}/label_maps.json", "w", encoding="utf-8") as f:
    json.dump(
        {"label2id": LABEL2ID, "id2label": ID2LABEL, "en_labels": EN_LABELS},
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved to ./{SAVE_DIR}")

In [ ]:
# Optional: download the folder from Colab
from google.colab import files
import shutil

shutil.make_archive("model", "zip", SAVE_DIR)
files.download("model.zip")

## Phase 9 — Try predictions

Quick sanity check before we build the Streamlit app.

In [ ]:
def predict(text, top_k=3):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        model.cuda()
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
    top = probs.topk(top_k)
    for score, idx in zip(top.values, top.indices):
        i = idx.item()
        print(f"{EN_LABELS[i]:20} ({ID2LABEL[i]})  {score.item():.2%}")


sample = raw["test"][10]
print("Headline:", sample["headline"])
print("True category:", sample["category"], "→", EN_LABELS[LABEL2ID[sample["category"]]])
print("\nPredictions:")
predict(f"{sample['headline']}. {sample['article'][:500]}")

---

### Optional — Push to Hugging Face Hub (free hosting for the web app)

Uncomment below, create a free account at [huggingface.co](https://huggingface.co), and get an access token.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
#
# HF_REPO = "your-username/amharic-news-classifier"  # change this
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)